# Choix des topics exposés dans l'outil de visualisation

**Question** : quels arbres l'outil expose-t-il par défaut, et pourquoi ?L'interface promet une sélection à 4 niveaux (racine → grand-parent → parent →enfant). Ce notebook mesure quels arbres peuvent tenir cette promesse, etdocumente le choix retenu.Deux notions distinctes :
- **propre** = topic relié et sans cycle (qualité des liens) ;
- **complet** = arbre possédant les 4 niveaux (profondeur maximale).

## Chargement et reconstruction (mêmes règles que l'outil)

In [1]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

DATA = Path("data")
topics = json.loads((DATA / "analysis/structure/taxonomy_3.json").read_text())["topics"]
docs = json.loads((DATA / "analysis/label/instances.json").read_text())["documents"]

# R1 — résolution canonique des homonymes : on retient l'UUID le plus petit
by_name = {}
for t in sorted(topics, key=lambda t: t["id"]):
    by_name.setdefault(t["name"], t)

own = Counter(lab["name"] for d in docs for lab in d["labels"])
TOTAL = sum(own.values())

print(f"{len(topics)} topics bruts -> {len(by_name)} noms uniques | {TOTAL} détections")

4840 topics bruts -> 4816 noms uniques | 9154 détections


In [2]:
# filtre qualité : on écarte isolés et cycles
est_parent = {t["parent"] for t in topics if t["parent"] in by_name}
isoles = {n for n, t in by_name.items() if not t["parent"] and n not in est_parent}


def cyclique(nom, vus=None):
    vus = vus or set()
    if nom in vus:
        return True
    vus.add(nom)
    p = by_name[nom]["parent"]
    return cyclique(p, vus) if p and p in by_name else False


cycliques = {n for n in by_name if cyclique(n)}
propre = set(by_name) - isoles - cycliques

parent_de = {n: by_name[n]["parent"] for n in propre if by_name[n]["parent"] in propre}
enfants = defaultdict(list)
for n, p in parent_de.items():
    enfants[p].append(n)

print(f"propres : {len(propre)}  (= {len(by_name)} uniques − {len(isoles)} isolés − {len(cycliques)} en cycle)")

propres : 3709  (= 4816 uniques − 1029 isolés − 78 en cycle)


**« Propre » ne dit rien de la profondeur** : un arbre à 2 nœuds est « propre ».La complétude est une propriété différente, mesurée ci-dessous.

## Profondeur des arbres

In [3]:
# hauteur d'un nœud = distance aux feuilles ; racines d'arbres = nœuds sans parent
memo = {}


def hauteur(n):
    if n not in memo:
        memo[n] = 0 if not enfants[n] else 1 + max(hauteur(c) for c in enfants[n])
    return memo[n]


def rec(n, vus=None):
    vus = vus or set()
    if n in vus:
        return 0
    vus.add(n)
    return own.get(n, 0) + sum(rec(c, vus) for c in enfants[n])


racines = [n for n in propre if n not in parent_de and enfants[n]]
lignes = [{"racine": r, "hauteur": hauteur(r), "topics": None, "détections": rec(r)}
          for r in racines]


def taille(r):
    total, pile = 0, [r]
    while pile:
        n = pile.pop()
        total += 1
        pile.extend(enfants[n])
    return total


for ligne in lignes:
    ligne["topics"] = taille(ligne["racine"])

arbres = pd.DataFrame(lignes).sort_values("détections", ascending=False)
arbres.groupby("hauteur").agg(
    nb_arbres=("racine", "count"),
    topics=("topics", "sum"),
    détections=("détections", "sum"),
).assign(pct_détections=lambda d: (100 * d["détections"] / TOTAL).round(0).astype(int))

,nb_arbres,topics,détections,pct_détections
hauteur,,,,
1,172,1032,1932,21
2,34,1336,2517,27
3,8,1341,3381,37


**Lecture** : hauteur 3 = les 4 niveaux présents (racine → grand-parent → parent →enfant) = **arbre complet**. Seuls 8 arbres le sont — mais ils portent à eux seuls**37 % des détections**. Les 143 arbres de hauteur 1 sont des étoiles plates(un parent + des feuilles) : dans une cascade à 4 sélecteurs, 2 menus resteraientvides — l'outil semblerait cassé.

## Les 8 arbres complets

In [4]:
arbres[arbres["hauteur"] == 3][["racine", "topics", "détections"]].reset_index(drop=True)

,racine,topics,détections
0,politiques sociales et redistribution,394,1374
1,gouvernance publique et pouvoirs publics,232,898
2,politiques environnementales et écologiques,243,448
3,organisation de l’État et fonction publique,153,359
4,état de droit et garanties fondamentales,94,87
5,politique migratoire et intégration,98,76
6,stratégies nationales et souveraineté,36,71
7,politiques de santé et bien-être collectif,91,68


Ce sont précisément **les grandes préoccupations** : politiques sociales,gouvernance publique, environnement, organisation de l'État… Les structures lesplus abouties de la taxonomie sont aussi les sujets les plus massifs.

## Les 3 options de filtre et leur couverture

In [5]:
options = []
for label, hmin in [("Arbres complets (4 niveaux)", 3), ("Arbres ≥ 3 niveaux", 2), ("Tous les arbres", 0)]:
    sel = arbres[arbres["hauteur"] >= hmin]
    options.append({
        "filtre": label,
        "arbres": len(sel),
        "topics": int(sel["topics"].sum()),
        "% détections": round(100 * sel["détections"].sum() / TOTAL),
    })
pd.DataFrame(options)

,filtre,arbres,topics,% détections
0,Arbres complets (4 niveaux),8,1341,37
1,Arbres ≥ 3 niveaux,42,2677,64
2,Tous les arbres,214,3709,86


## Le choix retenu (implémenté dans `simulation_graph.py`)
1. **Défaut = arbres complets.** L'interface tient sa promesse : chaque sélection   déroule le niveau suivant, aucun menu vide, tous les nœuds accessibles.   On montre le travail le plus abouti de l'équipe analyse — 37 % des détections,   et les sujets majeurs.
2. **Le reste n'est pas masqué, il est compté** : la bannière affiche « N arbres   en cours de structuration », accessibles via le filtre **Profondeur**   (`≥ 3 niveaux` : 64 % · `tous` : 86 %).
3. **La vue d'ensemble est le squelette connecté** (racines + grands-parents +   parents, avec arêtes, Fruchterman-Reingold) — un vrai graphe lisible, jamais   les 3221 feuilles d'un coup.**Ce qu'on attend de la prochaine livraison** : si la passe de structurationsuivante rattache les arbres de hauteur 1-2 aux racines existantes, la couverturedu défaut montera mécaniquement — le filtre est déjà prêt, aucun code à changer.